In [ ]:
## for each mock_community and v_regions, use rep_seq.fasta file to generate classification results
# requires specification of reference sequences and taxonomies for each V-region and models for the tools if require.
# if models are not provided, it will try to generate one with default params of the tool (which is not ideal)
# different params can be used for each tool to generate params grid and classification can be generated for each combination separately.

import os
import sys
import random
import shutil
import glob
from joblib import Parallel, delayed
from os.path import join, isfile, isdir, basename, exists, expandvars, dirname, splitext
from pathlib import Path
from collections import defaultdict
import itertools
import numpy as np
import pandas as pd
import src.utils
import src.generate_commands

In [ ]:
## set path vars
n_cpus = os.cpu_count() - 1
n_jobs = 1 # number_of_jobs to run in parallel
analysis_name = "mock-community"
project_dir = expandvars("/home/kardokhk/taxonova_bench/taxobench_five")
conda_dir = join(project_dir, "conda")
scripts_dir = join(project_dir, f"scripts/src")
reference_database_dir = join(project_dir, f"benchmark/data/ref_dbs")
mock_dir = join(project_dir, f"benchmark/data", analysis_name)



In [ ]:
## name of the mock communities
mock_names = ['gutmock', 'vaginalmock', 'mockrobiota-16S', 'mockrobiota-ITS']

## amplicon regions
v_regions = ['V4', 'V3-V4', 'ITS1', 'ITS2', 'V1-V9', 'ITS12']

## mock to v_region mapping
mock_v_regions = {
    'gutmock': ['V4', 'V3-V4', 'V1-V9'],
    'vaginalmock': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-16S': ['V4', 'V3-V4', 'V1-V9'],
    'mockrobiota-ITS': ['ITS1', 'ITS2', 'ITS12']
}



## Reference database or models needed for classification
## If model is required for classifications, then they should be provided up-front.
## If not found it will try to train one
# common databases contains reference sequence (.fasta) and taxonomy files (.tsv)
common_database = {}
for v_region in v_regions:
    v_region_dir = join(reference_database_dir, v_region)
    if isdir(v_region_dir):
        merged_file = join(v_region_dir, "merged.fasta")
        seq_file = join(v_region_dir, "sequences.fasta")
        taxa_file = join(v_region_dir, "taxonomy.tsv")

        if isfile(seq_file) and isfile(taxa_file):
            common_database[v_region] = {
                'seqs': seq_file,
                'taxa': taxa_file,
                'merged': merged_file if isfile(merged_file) else None
            }


# Tool-specific models
reference_database = {
    'taxonova': {},
    'vsearch': {},
    'blast': {},
    'naive-bayes': {},
    'idtaxa': {},
    'spingo': {}
}

for v_region, files in common_database.items():
    v_region_dir = join(reference_database_dir, v_region)
    
    # Initialize model paths
    taxonova_model = None
    nb_model = None
    idtaxa_model = None
    spingo_database = None

    # Search for taxonova model files
    taxonova_dir = join(v_region_dir, 'taxonova')
    if isdir(taxonova_dir):
        models = glob.glob(join(taxonova_dir, '*model.pt'))
        if models:
            taxonova_model = models[0]

    # Search for naive-bayes model files
    nb_dir = join(v_region_dir, 'naive-bayes')
    if isdir(nb_dir):
        nb_models = glob.glob(join(nb_dir, '*sklearn*qza'))
        if nb_models:
            nb_model = nb_models[0]

    # Search for idTaxa model files
    idtaxa_dir = join(v_region_dir, 'idtaxa')
    if isdir(idtaxa_dir):
        idtaxa_models = glob.glob(join(idtaxa_dir, '*.rds')) + glob.glob(join(idtaxa_dir, '*.RData'))
        if idtaxa_models:
            idtaxa_model = idtaxa_models[0]

    # Search for SPINGO database files
    spingo_dir = join(v_region_dir, 'spingo')
    if isdir(spingo_dir):
        spingo_db_files = glob.glob(join(spingo_dir, '*.fasta'))
        if spingo_db_files:
            spingo_database = spingo_db_files[0]

    # Populate reference_database
    reference_database['taxonova'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa'],
        'merged': files['merged'],
        'model': taxonova_model
    }
    reference_database['naive-bayes'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa'],
        'model': nb_model
    }
    reference_database['vsearch'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa']
    }
    reference_database['blast'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa']
    }
    reference_database['idtaxa'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa'],
        'model': idtaxa_model
    }
    reference_database['spingo'][v_region] = {
        'seqs': files['seqs'],
        'taxa': files['taxa'],
        'database': spingo_database
    }



## prepare combinations of methods and params to run for classification
method_parameters_combinations = {
    'taxonova': {'species_threshold': [0.5, 0.6, 0.7, 0.8],
                 'genus_threshold': [0.4, 0.5, 0.6],
                 'family_threshold': [0.3, 0.4],
                 'order_threshold': [0.2],
                 'class_threshold': [0.1],
                 'phylum_threshold': [0.1],
                 'domain_threshold': [0.1]
                },
    'vsearch': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'blast': {'p-maxaccepts': [1, 10, 100],
                'p-perc-identity': [0.80, 0.90, 0.97, 0.99],
                'p-min-consensus': [0.51, 0.75, 0.99]
                },
    'naive-bayes': {'p-confidence': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
                },
    'idtaxa': {'p-confidence': [30, 40, 50, 60, 70, 80, 90]
                },
    'spingo': {'kmersize': [8],
               'bootstrap': [10, 20, 30, 40, 50, 60, 70, 80, 100],
               'subsample': [10]
                },                                       
}

## prepare methods and conda_env combination (if needed separate activation per tools)
method_conda_combinations = {
    'taxonova': '/home/kardokhk/miniforge3/envs/mock-taxonomy',
    'vsearch': '/home/kardokhk/miniforge3/envs/mock-taxonomy',
    'blast': '/home/kardokhk/miniforge3/envs/mock-taxonomy',
    'naive-bayes': '/home/kardokhk/miniforge3/envs/mock-taxonomy',
    'idtaxa': '/home/kardokhk/miniforge3/envs/mock-taxonomy',
    'spingo': {
        'spingo_env': '/home/kardokhk/miniforge3/envs/spingo-env',  # Environment for SPINGO tool
        'processing_env': '/home/kardokhk/miniforge3/envs/mock-taxonomy'  # Environment for Python processing
    },
}


In [ ]:
## get rep_seqs and feature tables files from v_regions
## check dirs
rep_seq_files, feature_table_files = {}, {}
for v_region in v_regions:
    rep_seq_files[v_region], feature_table_files[v_region] = {}, {}
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        mockdir = join(mock_dir, mock_name, v_region, 'source')
        # Add rep_seqs_file
        rep_seq_glob = glob.glob(join(mockdir, "*", f'rep_seqs_{mock_name}.fasta'))
        rep_seq_glob = os.path.splitext(rep_seq_glob[0])[0]
        rep_seq_files[v_region][mock_name] = rep_seq_glob

        # Add feature_table_file
        feat_table_glob = glob.glob(join(mockdir, "*", f'table_{mock_name}.tsv'))
        feat_table_glob = os.path.splitext(feat_table_glob[0])[0]
        feature_table_files[v_region][mock_name] = feat_table_glob

rep_seq_files

In [ ]:
## to only generate commands and not actual run it, use dry_run
dry_run = False

commands = []
## Iterate over each methods and all combinations of its parameters and assign taxonomy
for v_region in v_regions:
    for mock_name in mock_names:
        # skip if region to mock mismatch
        if v_region not in mock_v_regions[mock_name]:
            continue
        
        rep_seq = rep_seq_files[v_region][mock_name]
        for method, param_grid in method_parameters_combinations.items():
            # get vars
            param_names = list(param_grid.keys())
            param_values = [v if isinstance(v, list) else [v] for v in param_grid.values()]

            # run each combinations of params
            for combination in itertools.product(*param_values):
                param_dict = dict(zip(param_names, combination))

                # generate outdir based on params_combo
                parameter_comb_id = ":".join(str(v) for v in param_dict.values())
                outdir = join(mock_dir, mock_name, v_region, 'classify', method, parameter_comb_id)

                ## Run taxonova
                if method == "taxonova":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):
                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_taxonova_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            scripts_dir=scripts_dir,
                            conda_env=conda_env,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Run vsearch
                if method == "vsearch":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):
                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_vsearch_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            conda_env=conda_env,
                            n_threads=n_cpus,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Run blast+
                if method == "blast":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):
                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_blast_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            conda_env=conda_env,
                            n_threads=n_cpus,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Run naive-bayes
                if method == "naive-bayes":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):

                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_naive_bayes_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            conda_env=conda_env,
                            n_threads=n_cpus,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Run idTaxa
                if method == "idtaxa":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):
                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_idtaxa_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            scripts_dir=scripts_dir,
                            conda_env=conda_env,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Run SPINGO
                if method == "spingo":
                    if not all(os.path.exists(f) for f in [join(outdir, 'taxonomy_final.tsv')]):
                        conda_env = method_conda_combinations.get(method)
                        cmd = src.generate_commands.build_spingo_command(
                            rep_seq_base=rep_seq,
                            outdir=outdir,
                            param_dict=param_dict,
                            reference_dict=reference_database[method][v_region],
                            scripts_dir=scripts_dir,
                            conda_env=conda_env,
                            n_threads=n_cpus,
                            dry_run=dry_run
                        )
                        commands.append(cmd)

                ## Likewise add cmds for other tools

## run in parallel
if not dry_run:
    Parallel(n_jobs=n_jobs) (delayed(os.system)(command) for command in list(commands))

In [ ]:
commands